# 🚁 Unsupervised Drone Telemetry Exploratory Data Analysis & Visualizations

This notebook provides visual analysis of normal vs spoofed drone telemetry to explore physical kinematic signatures, noise textures, and cross-correlations.

### Key Explorations:
1. **Dataset Overview & Statistical Summaries**: Class distributions across Normal DJI, Real ESP32, and Simulated Attacks.
2. **Distribution Density Grids & Boxplots**: Statistical distributions of kinematic and noise texture features.
3. **2D & 3D Spatial Flight Trajectories**: Real flight tracks vs simulated attacks and stationary ESP32 broadcast clusters.
4. **Physics-Based Kinematic Consistency Over Time**: Prediction error, acceleration dynamics, and speed-turn rate correlations.
5. **Unsupervised Manifold Embeddings (PCA & t-SNE)**: 2D projections explaining cluster separability for unsupervised anomaly detection models.
6. **Aerodynamic Coupling & Cross-Correlation Heatmaps**: Physical feature correlation breakdown under spoofing.

## 1. Setup Working Directory & Imports

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from implement.workflows.per_class_evaluation import get_labeled_datasets
from implement.utils.helper.features import CROSS_CORRELATION_16_FEATURES
from implement.utils.helper import get_output_dir

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"Operating Directory: {os.getcwd()}")
print(f"Features Available: {CROSS_CORRELATION_16_FEATURES}")

## 2. Load & Prepare Datasets

In [ ]:
print("Loading preprocessed telemetry datasets...")
dji_df, esp32_df = get_labeled_datasets(features=CROSS_CORRELATION_16_FEATURES)

combined_df = pd.concat([dji_df, esp32_df], ignore_index=True)

print(f"Total Telemetry Records: {len(combined_df):,}")
print(f"Normal DJI Records:     {len(dji_df):,} ({len(dji_df['flight_id'].unique())} flights)")
print(f"Spoofed Records:        {len(esp32_df):,} ({len(esp32_df['flight_id'].unique())} flights)")

print("\nPer-Class Breakdown:")
class_counts = combined_df['attack_class'].value_counts()
display(pd.DataFrame({'Record Count': class_counts, 'Percentage (%)': (class_counts / len(combined_df) * 100).round(2)}))

## 3. Feature Distribution Density Grids
Compare continuous probability densities between Normal DJI, Real ESP32, and Simulated Attacks.

In [ ]:
features_to_plot = [
    'ground_speed', 'vertical_speed', 'acceleration', 'turn_rate',
    'prediction_error', 'speed_spectral_entropy', 'corr_speed_turn', 'corr_accel_turn'
]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

palette = {'Normal DJI': '#2ecc71', 'Real ESP32': '#e74c3c', 'Sim Baseline': '#9b59b6', 'Sim Easy': '#e67e22', 'Sim Hard': '#34495e', 'Sim Medium': '#f1c40f', 'Sim Geometry': '#1abc9c'}

for i, feat in enumerate(features_to_plot):
    ax = axes[i]
    for cls in ['Normal DJI', 'Real ESP32', 'Sim Easy', 'Sim Hard', 'Sim Geometry']:
        sub = combined_df[combined_df['attack_class'] == cls][feat].dropna()
        if len(sub) > 0:
            sns.kdeplot(sub, ax=ax, label=cls, color=palette.get(cls, '#7f8c8d'), fill=True, alpha=0.15, linewidth=1.8)
    ax.set_title(f"Distribution: {feat}", fontsize=11, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel("Density")
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Multi-Class Boxplot Distributions
Shows median, interquartile ranges (IQR), and anomalies across individual attack types.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 14))
axes = axes.flatten()

box_features = ['ground_speed', 'acceleration', 'path_curvature', 'prediction_error', 'corr_speed_turn', 'corr_vert_speed']

for i, feat in enumerate(box_features):
    ax = axes[i]
    sns.boxplot(data=combined_df, x='attack_class', y=feat, ax=ax, palette=palette, showfliers=False)
    ax.set_title(f"{feat} across Attack Classes", fontsize=11, fontweight='bold')
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

## 5. 2D Spatial Trajectories
Visualizing geographic trajectories (Latitude vs Longitude) reveals the distinct patterns of real UAV flights vs stationary spoofer noise and synthetic geometric flight paths.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

sample_classes = ['Normal DJI', 'Real ESP32', 'Sim Baseline', 'Sim Easy', 'Sim Hard', 'Sim Geometry']

for i, cls in enumerate(sample_classes):
    ax = axes[i]
    sub = combined_df[combined_df['attack_class'] == cls]
    flight_sample = sub['flight_id'].unique()[:3]
    for fid in flight_sample:
        fl_data = sub[sub['flight_id'] == fid]
        if 'latitude' in fl_data.columns and 'longitude' in fl_data.columns:
            ax.plot(fl_data['longitude'], fl_data['latitude'], '.-', alpha=0.7, markersize=3, label=str(fid)[:15])
    ax.set_title(f"{cls} Spatial Path", fontsize=11, fontweight='bold')
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(fontsize=8, loc='best')

plt.tight_layout()
plt.show()

## 6. Physics-Based Kinematic Consistency Over Time
Plotting `prediction_error` and `corr_speed_turn` over individual flight time series.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=False)

# Normal DJI Flight Sample
normal_sample = combined_df[combined_df['attack_class'] == 'Normal DJI']
norm_fid = normal_sample['flight_id'].unique()[0]
norm_flight = normal_sample[normal_sample['flight_id'] == norm_fid].reset_index(drop=True)

# Spoofed Flight Sample
spoo_sample = combined_df[combined_df['attack_class'] == 'Real ESP32'].reset_index(drop=True)

ax1.plot(norm_flight.index[:250], norm_flight['prediction_error'][:250], color='#2ecc71', label='Normal DJI (Physical Motion)', linewidth=2)
ax1.plot(spoo_sample.index[:250], spoo_sample['prediction_error'][:250], color='#e74c3c', label='Real ESP32 (Spoofed Stationary Noise)', linewidth=2, alpha=0.8)
ax1.set_title("Kinematic Prediction Error Over Time (First 250 Samples)", fontweight='bold')
ax1.set_ylabel("Prediction Error")
ax1.legend()

ax2.plot(norm_flight.index[:250], norm_flight['corr_speed_turn'][:250], color='#2ecc71', label='Normal DJI (Speed-Turn Correlation)', linewidth=2)
ax2.plot(spoo_sample.index[:250], spoo_sample['corr_speed_turn'][:250], color='#e74c3c', label='Real ESP32 (Speed-Turn Correlation)', linewidth=2, alpha=0.8)
ax2.set_title("Rolling Speed-Turn Correlation Over Time", fontweight='bold')
ax2.set_ylabel("Corr(Speed, Turn)")
ax2.set_xlabel("Sample Index (Time Step @ 2Hz)")
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Unsupervised Manifold Projections: 2D PCA & t-SNE
Projects high-dimensional telemetry into 2D space to visualize separation between Normal DJI and attack classes.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Subsample for clean projection
sample_df = combined_df.groupby('attack_class', group_keys=False).apply(lambda x: x.sample(min(len(x), 800), random_state=42))
X_sub = sample_df[CROSS_CORRELATION_16_FEATURES].fillna(0.0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sub)

# 2D PCA
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

# 2D t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for cls in sample_df['attack_class'].unique():
    mask = (sample_df['attack_class'].values == cls)
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], label=cls, color=palette.get(cls, '#7f8c8d'), alpha=0.6, s=18)
    ax2.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=cls, color=palette.get(cls, '#7f8c8d'), alpha=0.6, s=18)

ax1.set_title(f"2D PCA Projection (Exp. Var: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%)", fontweight='bold')
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")
ax1.legend(loc='best', fontsize=8)

ax2.set_title("2D t-SNE Manifold Embedding", fontweight='bold')
ax2.set_xlabel("t-SNE 1")
ax2.set_ylabel("t-SNE 2")
ax2.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Feature Cross-Correlation Heatmaps
Shows correlation matrices for Normal DJI vs Spoofed telemetry.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

corr_normal = dji_df[CROSS_CORRELATION_16_FEATURES].corr()
corr_spoofed = esp32_df[CROSS_CORRELATION_16_FEATURES].corr()

sns.heatmap(corr_normal, ax=ax1, cmap='coolwarm', vmin=-1, vmax=1, annot=False, cbar=True)
ax1.set_title("Normal DJI Physical Feature Correlations", fontsize=12, fontweight='bold')

sns.heatmap(corr_spoofed, ax=ax2, cmap='coolwarm', vmin=-1, vmax=1, annot=False, cbar=True)
ax2.set_title("Spoofed Telemetry Feature Correlations", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()